In [2]:
import pandas as pd
import folium
import math
from IPython.display import display

# ==========================================
# 1. DATA LOKASI FASILITAS
# ==========================================

data = {
    "Nama": [
        "Universitas Bengkulu",
        "Rumah Sakit",
        "Perpustakaan",
        "Kantor Pemerintahan",
        "Stasiun Pengisian BBM"
    ],
    "Kategori": [
        "Pendidikan",
        "Kesehatan",
        "Pendidikan",
        "Pemerintahan",
        "Energi"
    ],
    "Latitude": [
        -3.7589,
        -3.7920,
        -3.7580,
        -3.8000,
        -3.7800
    ],
    "Longitude": [
        102.2769,
        102.2650,
        102.2700,
        102.2800,
        102.2900
    ]
}

df = pd.DataFrame(data)

# ==========================================
# 2. VALIDASI DATA
# ==========================================

kolom_wajib = ["Nama", "Kategori", "Latitude", "Longitude"]

if df.empty:
    raise ValueError("Data lokasi masih kosong.")

if df[kolom_wajib].isnull().any().any():
    raise ValueError("Data memiliki nilai kosong.")

df["Latitude"] = pd.to_numeric(
    df["Latitude"], errors="raise"
)

df["Longitude"] = pd.to_numeric(
    df["Longitude"], errors="raise"
)

for _, lokasi in df.iterrows():
    if not (
        math.isfinite(lokasi["Latitude"])
        and math.isfinite(lokasi["Longitude"])
    ):
        raise ValueError("Koordinat harus berupa angka yang valid.")

    if not -90 <= lokasi["Latitude"] <= 90:
        raise ValueError("Latitude harus berada antara -90 dan 90.")

    if not -180 <= lokasi["Longitude"] <= 180:
        raise ValueError("Longitude harus berada antara -180 dan 180.")

if df["Nama"].duplicated().any():
    raise ValueError("Nama lokasi tidak boleh duplikat.")

print("DATA LOKASI FASILITAS")
display(df)


# ==========================================
# 3. FUNGSI MENGHITUNG JARAK
#    MENGGUNAKAN RUMUS HAVERSINE
# ==========================================

def hitung_jarak(lat1, lon1, lat2, lon2):
    radius_bumi = 6371.0  # kilometer

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    selisih_lat = lat2 - lat1
    selisih_lon = lon2 - lon1

    a = (
        math.sin(selisih_lat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(selisih_lon / 2) ** 2
    )

    # Mencegah kesalahan pembulatan floating point
    a = max(0.0, min(1.0, a))

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return radius_bumi * c


# ==========================================
# 4. MENENTUKAN LOKASI ACUAN
# ==========================================

nama_acuan = "Universitas Bengkulu"

data_acuan = df[df["Nama"] == nama_acuan]

if data_acuan.empty:
    raise ValueError("Lokasi acuan tidak ditemukan.")

acuan = data_acuan.iloc[0]

# Hitung jarak seluruh fasilitas dari lokasi acuan
df["Jarak (km)"] = df.apply(
    lambda baris: hitung_jarak(
        acuan["Latitude"],
        acuan["Longitude"],
        baris["Latitude"],
        baris["Longitude"]
    ),
    axis=1
)

# ==========================================
# 5. ANALISIS HASIL JARAK
# ==========================================

print("\nHASIL PERHITUNGAN JARAK")

hasil = df[
    ["Nama", "Kategori", "Jarak (km)"]
].sort_values("Jarak (km)")

display(hasil.round(3).reset_index(drop=True))

# Lokasi terdekat tidak termasuk lokasi acuan
kandidat = df[df["Nama"] != nama_acuan].copy()

if kandidat.empty:
    raise ValueError(
        "Diperlukan setidaknya satu fasilitas selain lokasi acuan."
    )

terdekat = kandidat.loc[kandidat["Jarak (km)"].idxmin()]

print("\nHASIL ANALISIS GIS")
print("Lokasi acuan :", nama_acuan)
print("Fasilitas terdekat :", terdekat["Nama"])
print("Kategori :", terdekat["Kategori"])
print("Jarak garis lurus :", round(terdekat["Jarak (km)"], 3), "km")


# ==========================================
# 6. MEMBUAT PETA INTERAKTIF
# ==========================================

peta = folium.Map(
    location=[
        acuan["Latitude"],
        acuan["Longitude"]
    ],
    zoom_start=13,
    tiles="OpenStreetMap"
)

# Tambahkan marker setiap lokasi
for _, lokasi in df.iterrows():

    if lokasi["Nama"] == nama_acuan:
        warna = "blue"
    elif lokasi["Nama"] == terdekat["Nama"]:
        warna = "green"
    else:
        warna = "red"

    popup_html = f"""
    <b>{lokasi['Nama']}</b><br>
    Kategori: {lokasi['Kategori']}<br>
    Latitude: {lokasi['Latitude']}<br>
    Longitude: {lokasi['Longitude']}<br>
    Jarak dari lokasi acuan: {lokasi['Jarak (km)']:.3f} km
    """

    folium.Marker(
        location=[
            lokasi["Latitude"],
            lokasi["Longitude"]
        ],
        tooltip=lokasi["Nama"],
        popup=folium.Popup(popup_html, max_width=300),
        icon=folium.Icon(
            color=warna,
            icon="info-sign"
        )
    ).add_to(peta)

# Tambahkan garis dari lokasi acuan ke fasilitas terdekat
folium.PolyLine(
    locations=[
        [acuan["Latitude"], acuan["Longitude"]],
        [terdekat["Latitude"], terdekat["Longitude"]]
    ],
    color="green",
    weight=4,
    opacity=0.8,
    tooltip="Garis lurus menuju fasilitas terdekat"
).add_to(peta)

# Sesuaikan tampilan peta agar seluruh lokasi terlihat
batas_peta = [
    [baris["Latitude"], baris["Longitude"]]
    for _, baris in df.iterrows()
]

peta.fit_bounds(batas_peta)

# ==========================================
# 7. MENAMPILKAN PETA
# ==========================================

print("\nPETA INTERAKTIF GIS")
print("Marker biru  : lokasi acuan")
print("Marker hijau : fasilitas terdekat")
print("Marker merah : fasilitas lainnya")

display(peta)

DATA LOKASI FASILITAS


,Nama,Kategori,Latitude,Longitude
0,Universitas Bengkulu,Pendidikan,-3.7589,102.2769
1,Rumah Sakit,Kesehatan,-3.7920,102.2650
2,Perpustakaan,Pendidikan,-3.7580,102.2700
3,Kantor Pemerintahan,Pemerintahan,-3.8000,102.2800
4,Stasiun Pengisian BBM,Energi,-3.7800,102.2900



HASIL PERHITUNGAN JARAK


,Nama,Kategori,Jarak (km)
0,Universitas Bengkulu,Pendidikan,0.000
1,Perpustakaan,Pendidikan,0.772
2,Stasiun Pengisian BBM,Energi,2.760
3,Rumah Sakit,Kesehatan,3.910
4,Kantor Pemerintahan,Pemerintahan,4.583



HASIL ANALISIS GIS
Lokasi acuan : Universitas Bengkulu
Fasilitas terdekat : Perpustakaan
Kategori : Pendidikan
Jarak garis lurus : 0.772 km

PETA INTERAKTIF GIS
Marker biru  : lokasi acuan
Marker hijau : fasilitas terdekat
Marker merah : fasilitas lainnya
